# Test środowiska
Uruchom z włączonym profilem `analysis` (Spark, HBase). Komórka Kafki wymaga też profilu `ingest`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test-srodowiska").enableHiveSupport().getOrCreate()
spark.sql("SHOW DATABASES").show()

df = spark.createDataFrame([(1, "steam"), (2, "wiki")], ["id", "zrodlo"])
df.write.mode("overwrite").format("orc").saveAsTable("default.test_spark")
spark.sql("SELECT * FROM default.test_spark").show()

In [ ]:
import happybase

conn = happybase.Connection("hbase", 9090)
if b"test" not in conn.tables():
    conn.create_table("test", {"cf": dict()})
t = conn.table("test")
t.put(b"553850#20240505", {b"cf:neg": b"48466", b"cf:pos": b"65690"})
print(t.row(b"553850#20240505"))
conn.close()

In [ ]:
from kafka import KafkaProducer, KafkaConsumer
import json

producer = KafkaProducer(bootstrap_servers="kafka:9092",
                         value_serializer=lambda v: json.dumps(v).encode())
producer.send("test", {"msg": "dziala"})
producer.flush()

consumer = KafkaConsumer("test", bootstrap_servers="kafka:9092",
                         auto_offset_reset="earliest", consumer_timeout_ms=5000)
for m in consumer:
    print(m.value)